# Scraping CampusNest (Streamlit) with a browser

This notebook shows a simple approach to scraping the tutor and housing listing data from the public Streamlit site:

- https://campusnest.streamlit.app/

Why a browser-based scraper?

Streamlit apps render much of their UI dynamically. For many Streamlit sites, `requests.get(url).text` does **not** contain the final catalogue data you see in the browser. So we use **Playwright** (headless Chromium) to load the page, let it render, and then extract the DOM.

At the end, we save:

- `campusnest_tutors.csv`
- `campusnest_listings.csv`


## 0) Install dependencies (first time only)

Run the next cell once in your environment.


In [ ]:
!pip -q install playwright pandas
!playwright install chromium

## 1) Imports and configuration


In [ ]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
from playwright.sync_api import sync_playwright

BASE_URL = "https://campusnest.streamlit.app/"

## 2) Helper functions

We will:

1. Load the home page
2. Discover internal page links (Tutoring, Student Housing)
3. Extract the card text (`div.sh-card`)
4. Parse structured fields using regular expressions


In [ ]:
def discover_internal_pages(page, base_url: str) -> dict:
    """Return a mapping {name: url} for internal pages we can see in the UI."""
    page.goto(base_url, wait_until="networkidle")
    time.sleep(1.0)

    links = page.locator("a").all()
    found = {}
    for a in links:
        href = a.get_attribute("href")
        label = (a.inner_text() or "").strip()
        if not href:
            continue
        # Convert relative -> absolute
        abs_url = urljoin(base_url, href)
        if abs_url.startswith(base_url):
            # Keep a few likely navigation items
            if label:
                found[label] = abs_url

    return found


def pick_page_url(pages: dict, contains: str) -> str:
    """Pick the first discovered page URL whose label contains a substring (case-insensitive)."""
    contains_low = contains.lower()
    for label, url in pages.items():
        if contains_low in label.lower():
            return url
    raise RuntimeError(f"Could not find a page whose label contains: {contains}. Discovered labels: {list(pages.keys())[:20]}")


def set_number_input_by_label(page, label_text: str, value: int):
    """Set a Streamlit number input by its visible label (best-effort)."""
    # Streamlit often renders labels in a <label> element, and the input nearby.
    label = page.get_by_text(label_text, exact=True)
    # Navigate to the closest input[type=number] in the same container.
    container = label.locator("xpath=ancestor::div[1]")
    inp = container.locator("input[type='number']")
    if inp.count() == 0:
        # fallback: first numeric input on page
        inp = page.locator("input[type='number']").first
    inp.click()
    inp.fill(str(value))
    inp.press("Enter")
    time.sleep(0.8)


def extract_card_texts(page) -> list[str]:
    """Extract visible card text blocks from div.sh-card."""
    page.wait_for_timeout(500)
    cards = page.locator("div.sh-card")
    texts = []
    for i in range(cards.count()):
        t = cards.nth(i).inner_text().strip()
        if t:
            texts.append(t)
    return texts


def parse_tutor_card(card_text: str) -> dict:
    """Parse one tutor card block into structured fields."""
    # Example structure (approx):
    # Name · €26/h
    # Headline...
    # Subjects: ...
    # Languages: ...
    # ID: TUT-001

    lines = [ln.strip() for ln in card_text.splitlines() if ln.strip()]
    title = lines[0]
    m = re.match(r"^(?P<name>.+?)\s*·\s*€(?P<price>\d+)\/h$", title)
    if not m:
        # tolerate slightly different formatting
        m = re.match(r"^(?P<name>.+?)\s*·\s*€(?P<price>\d+)", title)
    name = m.group("name").strip() if m else title
    price = int(m.group("price")) if m and m.groupdict().get("price") else None

    headline = None
    subjects = None
    languages = None
    tutor_id = None

    for ln in lines[1:]:
        if ln.lower().startswith("subjects:"):
            subjects = ln.split(":", 1)[1].strip()
        elif ln.lower().startswith("languages:"):
            languages = ln.split(":", 1)[1].strip()
        elif ln.lower().startswith("id:"):
            tutor_id = ln.split(":", 1)[1].strip()
        else:
            # first non-field line becomes headline
            if headline is None:
                headline = ln

    return {
        "id": tutor_id,
        "name": name,
        "price_eur_per_hour": price,
        "headline": headline,
        "subjects": subjects,
        "languages": languages,
        "raw": card_text,
    }


def parse_listing_card(card_text: str) -> dict:
    """Parse one listing card block into structured fields."""
    # Example structure (approx):
    # Studio in Lambrate · €600/mo
    # Studio in Lambrate
    # Min stay: 6 months · Deposit: €800
    # ID: LIST-0002

    lines = [ln.strip() for ln in card_text.splitlines() if ln.strip()]
    title_line = lines[0]
    m = re.match(r"^(?P<title>.+?)\s*·\s*€(?P<rent>\d+)\/mo$", title_line)
    title = m.group("title").strip() if m else title_line
    rent = int(m.group("rent")) if m and m.groupdict().get("rent") else None

    listing_id = None
    listing_type = None
    area = None
    min_stay_months = None
    deposit_eur = None

    # Derive type/area from the title pattern "<Type> in <Area>"
    m2 = re.match(r"^(?P<type>.+?)\s+in\s+(?P<area>.+)$", title)
    if m2:
        listing_type = m2.group("type").strip()
        area = m2.group("area").strip()

    for ln in lines[1:]:
        if ln.lower().startswith("id:"):
            listing_id = ln.split(":", 1)[1].strip()
        if "min stay" in ln.lower():
            # Min stay: 6 months · Deposit: €800
            ms = re.search(r"min stay:\s*(\d+)\s*months", ln, flags=re.I)
            if ms:
                min_stay_months = int(ms.group(1))
            dep = re.search(r"deposit:\s*€(\d+)", ln, flags=re.I)
            if dep:
                deposit_eur = int(dep.group(1))

    return {
        "id": listing_id,
        "title": title,
        "type": listing_type,
        "area": area,
        "rent_eur_per_month": rent,
        "min_stay_months": min_stay_months,
        "deposit_eur": deposit_eur,
        "raw": card_text,
    }


## 3) Scrape tutors

We will:

1. Discover page URLs
2. Navigate to the Tutoring page
3. Loop over the pagination input (Page 1..N)
4. Extract cards and parse them


In [ ]:
tutor_rows = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()

    pages = discover_internal_pages(page, BASE_URL)
    tutoring_url = pick_page_url(pages, "Tutoring")

    page.goto(tutoring_url, wait_until="networkidle")
    time.sleep(1.0)

    seen_ids = set()
    # We don't know the page count in advance; loop until no new IDs appear.
    for page_num in range(1, 30):
        set_number_input_by_label(page, "Page", page_num)
        cards = extract_card_texts(page)
        if not cards:
            break

        new_this_page = 0
        for c in cards:
            row = parse_tutor_card(c)
            if row.get("id") and row["id"] not in seen_ids:
                tutor_rows.append(row)
                seen_ids.add(row["id"])
                new_this_page += 1

        # Stop when pagination no longer yields new results
        if new_this_page == 0:
            break

    browser.close()

tutors_df = pd.DataFrame(tutor_rows).drop_duplicates(subset=["id"]).reset_index(drop=True)
tutors_df.head(10), len(tutors_df)

## 4) Scrape housing listings

Same logic as tutors, but parsing listing card fields.


In [ ]:
listing_rows = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()

    pages = discover_internal_pages(page, BASE_URL)
    housing_url = pick_page_url(pages, "Student Housing")

    page.goto(housing_url, wait_until="networkidle")
    time.sleep(1.0)

    seen_ids = set()
    for page_num in range(1, 80):
        set_number_input_by_label(page, "Page", page_num)
        cards = extract_card_texts(page)
        if not cards:
            break

        new_this_page = 0
        for c in cards:
            row = parse_listing_card(c)
            if row.get("id") and row["id"] not in seen_ids:
                listing_rows.append(row)
                seen_ids.add(row["id"])
                new_this_page += 1

        if new_this_page == 0:
            break

    browser.close()

listings_df = pd.DataFrame(listing_rows).drop_duplicates(subset=["id"]).reset_index(drop=True)
listings_df.head(10), len(listings_df)

## 5) Cleaning

- Split subjects/languages into lists
- Keep raw text for debugging


In [ ]:
def split_csv_field(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return []
    return [x.strip() for x in str(s).split(",") if x.strip()]

tutors_df["subjects_list"] = tutors_df["subjects"].apply(split_csv_field)
tutors_df["languages_list"] = tutors_df["languages"].apply(split_csv_field)

tutors_df[["id","name","price_eur_per_hour","subjects_list","languages_list"]].head(10)

## 6) Save to CSV


In [ ]:
tutors_csv_path = "campusnest_tutors.csv"
listings_csv_path = "campusnest_listings.csv"

tutors_df.to_csv(tutors_csv_path, index=False)
listings_df.to_csv(listings_csv_path, index=False)

tutors_csv_path, listings_csv_path